# EXIOBASE Sweden hotspot analysis

This notebook reproduces the hotspot analysis at **whole-Sweden level** rather than splitting Sweden into Stockholm and Rest of Sweden.

It keeps the same three dimensions as the original notebook:
- material extraction
- air emissions / GHG
- economic value / factor inputs

The consumption-based analysis uses the Sweden final-demand column(s) directly from EXIOBASE after `calc_all()` has been run.

In [ ]:
import pymrio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging
import time

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
from IPython.display import display

In [ ]:

# =============================================================================
# Human-readable labels and diagnostics
# =============================================================================

import re
from textwrap import fill

EXIOBASE_REGION_NAMES = {
    "WA": "Rest of world: Asia and Pacific",
    "WE": "Rest of world: Europe",
    "WF": "Rest of world: Africa",
    "WL": "Rest of world: America",
    "WM": "Rest of world: Middle East",
}

GHG_UNIT_SCALE = 1e6  # convert kg -> kt; change to 1e3 if your file stores tonnes


def country_name(code):
    """Return a readable country/region name for an EXIOBASE code."""
    code = str(code).strip()
    if code in EXIOBASE_REGION_NAMES:
        return EXIOBASE_REGION_NAMES[code]

    try:
        import pycountry
        if len(code) == 2 and code.isalpha():
            c = pycountry.countries.get(alpha_2=code.upper())
            if c is not None:
                return c.name
    except Exception:
        pass

    return code


def sector_name(name):
    """Return the sector label as a full string, without truncation."""
    return str(name).strip()


def format_country_sector_pair(idx):
    """Format a MultiIndex pair (country, sector) with readable names."""
    return f"{country_name(idx[0])}: {sector_name(idx[1])}"


def resolve_rows(index, candidates):
    """
    Resolve row names robustly against exact matches and loose string matches.

    This helps when EXIOBASE row labels differ slightly between file versions.
    """
    index_strings = [str(x).strip() for x in index]
    resolved = []
    for cand in candidates:
        cand_str = str(cand).strip()
        if cand_str in index_strings:
            resolved.append(cand_str)
            continue

        cand_norm = re.sub(r"\s+", " ", cand_str.lower())
        for row in index_strings:
            row_norm = re.sub(r"\s+", " ", row.lower())
            if row_norm == cand_norm or cand_norm in row_norm or row_norm in cand_norm:
                resolved.append(row)
                break

    return list(dict.fromkeys(resolved))


In [ ]:
# Path to EXIOBASE 3 zip file (adjust to your local setup).
EXIOBASE_PATH = Path("C:/EXIOBASE3/IOT_2024_pxp.zip")
if not EXIOBASE_PATH.exists():
    raise FileNotFoundError(f"EXIOBASE file not found: {EXIOBASE_PATH}")

# Output directory for results.
OUTPUT_DIR = Path("./results_sweden")
OUTPUT_DIR.mkdir(exist_ok=True)

# Sweden's EXIOBASE country code.
SWEDEN_CODE = "SE"

# Base year.
BASE_YEAR = 2024

In [ ]:
def load_exiobase(path):
    """Load EXIOBASE 3 and compute x and A."""
    log.info(f"Loading EXIOBASE from {path} ...")
    t0 = time.time()

    exio = pymrio.parse_exiobase3(path=path)

    log.info(f"Parsed in {time.time() - t0:.1f}s. Computing x and A ...")
    exio.x = pymrio.calc_x(exio.Z, exio.Y)
    exio.A = pymrio.calc_A(exio.Z, exio.x)

    log.info(f"A matrix shape: {exio.A.shape}")
    log.info(f"Y matrix shape: {exio.Y.shape}")
    log.info(f"Regions: {len(exio.get_regions())}")
    log.info(f"Sectors per region: {len(exio.get_sectors())}")

    for ext_name in ["material", "air_emissions", "factor_inputs"]:
        ext = getattr(exio, ext_name, None)
        if ext is not None and ext.F is not None:
            log.info(f"Extension '{ext_name}': {ext.F.shape[0]} stressors")
        else:
            log.warning(f"Extension '{ext_name}': NOT FOUND")

    return exio


exio = load_exiobase(EXIOBASE_PATH)

In [ ]:

# =============================================================================
# Material rows -> Anthesis four primary categories
# =============================================================================

MATERIAL_CATEGORY_MAP = {
    "biomass": list(range(0, 23)),
    "fossil": list(range(23, 33)),
    "metals": list(range(33, 48)),
    "minerals": list(range(48, 62)),
    "recycled": [],
}


def aggregate_material_categories(F_material):
    """Aggregate the 62 EXIOBASE material rows into four Anthesis categories."""
    row_names = F_material.index.tolist()
    result = {}
    for category, row_indices in MATERIAL_CATEGORY_MAP.items():
        if not row_indices:
            continue
        selected_rows = [row_names[i] for i in row_indices]
        result[category] = F_material.loc[selected_rows].sum(axis=0)
    return pd.DataFrame(result).T


all_indices = []
for cat, indices in MATERIAL_CATEGORY_MAP.items():
    all_indices.extend(indices)
assert sorted(all_indices) == list(range(62)), "Material mapping does not cover all 62 rows"
assert len(all_indices) == len(set(all_indices)), "Material mapping has overlaps"
print("Material category mapping OK: 62 rows -> 4 categories")


# =============================================================================
# GHG emission rows -> aggregated GHG (CO2e)
# =============================================================================

GWP = {
    "CO2": 1,
    "CH4": 28,
    "N2O": 265,
    "SF6": 23500,
    "HFC": 1,
    "PFC": 1,
}

GHG_FOSSIL_ROWS = {
    "CO2": [
        "CO2 - combustion - air",
        "CO2 - non combustion - Cement production - air",
        "CO2 - non combustion - Lime production - air",
        "CO2 - waste - fossil - air",
    ],
    "CH4": [
        "CH4 - combustion - air",
        "CH4 - non combustion - Extraction/production of (natural) gas - air",
        "CH4 - non combustion - Extraction/production of crude oil - air",
        "CH4 - non combustion - Mining of antracite - air",
        "CH4 - non combustion - Mining of bituminous coal - air",
        "CH4 - non combustion - Mining of coking coal - air",
        "CH4 - non combustion - Mining of lignite (brown coal) - air",
        "CH4 - non combustion - Mining of sub-bituminous coal - air",
        "CH4 - non combustion - Oil refinery - air",
        "CH4 - agriculture - air",
        "CH4 - waste - air",
    ],
    "N2O": [
        "N2O - combustion - air",
        "N2O - agriculture - air",
    ],
    "SF6": ["SF6 - air"],
    "HFC": ["HFC - air"],
    "PFC": ["PFC - air"],
}

GHG_BIOGENIC_ROWS = [
    "CO2_bio - combustion - air",
    "CH4_bio - combustion - air",
    "N2O_bio - combustion - air",
    "CO2 - agriculture - peat decay - air",
    "CO2 - waste - biogenic - air",
]


def aggregate_ghg(F_air, scale_to_kt=GHG_UNIT_SCALE, verbose=False):
    """
    Aggregate air-emission rows into fossil and biogenic GHG in kt CO2e.

    If the results are zero or implausibly small, check:
    1) whether the row names in your EXIOBASE file differ from the candidates above;
    2) whether your file stores emissions in tonnes rather than kg.
    """
    all_rows = F_air.index.tolist()
    fossil_total = pd.Series(0.0, index=F_air.columns, dtype="float64")
    bio_total = pd.Series(0.0, index=F_air.columns, dtype="float64")

    matched = {}
    for gas, candidates in GHG_FOSSIL_ROWS.items():
        rows = resolve_rows(all_rows, candidates)
        matched[gas] = rows
        if not rows and verbose:
            print(f"  WARNING: no rows matched for {gas}")
        if rows:
            fossil_total += F_air.loc[rows].sum(axis=0) * GWP[gas]

    bio_rows = resolve_rows(all_rows, GHG_BIOGENIC_ROWS)
    matched["biogenic"] = bio_rows
    if not bio_rows and verbose:
        print("  WARNING: no rows matched for biogenic GHG")
    if bio_rows:
        for row_name in bio_rows:
            row_lower = str(row_name).lower()
            if "co2" in row_lower:
                bio_total += F_air.loc[row_name]
            elif "ch4" in row_lower:
                bio_total += F_air.loc[row_name] * 28
            elif "n2o" in row_lower:
                bio_total += F_air.loc[row_name] * 265

    if verbose:
        matched_counts = {k: len(v) for k, v in matched.items()}
        print("  Matched GHG rows:", matched_counts)
        print(f"  Raw fossil sum:   {fossil_total.sum():,.3g}")
        print(f"  Raw biogenic sum: {bio_total.sum():,.3g}")

    fossil_total = fossil_total / scale_to_kt
    bio_total = bio_total / scale_to_kt

    out = pd.DataFrame({
        "GHG_fossil_CO2e_kt": fossil_total,
        "GHG_biogenic_CO2e_kt": bio_total,
    }).T

    if (out.abs().sum().sum() == 0) or out.isna().all().all():
        print(
            "WARNING: aggregated GHG results are zero or missing. "
            "Check the row-name matching and the GHG_UNIT_SCALE setting."
        )

    return out


air_rows = exio.air_emissions.F.index.tolist()
missing = []
for gas, rows in GHG_FOSSIL_ROWS.items():
    for r in rows:
        if r not in air_rows:
            missing.append(r)
for r in GHG_BIOGENIC_ROWS:
    if r not in air_rows:
        missing.append(r)

if missing:
    print(f"WARNING: {len(missing)} row(s) not found:")
    for m in missing:
        print(f"  {m}")
else:
    print("GHG row mapping OK: all specified rows found")


In [ ]:
# Compute all account matrices needed for production-based and consumption-based analysis.
# This is the step that enables D_pba and D_cba on the extension accounts.
log.info("Running calc_all() ...")
exio.calc_all()
log.info("calc_all() complete.")

In [ ]:
def extract_sweden_results(exio):
    """Extract Sweden PBA and CBA accounts for the three dimensions."""
    results = {}

    # ------------------------------------------------------------------
    # 1) Material extraction
    # ------------------------------------------------------------------
    mat_ext = getattr(exio, "material", None)
    if mat_ext is not None and mat_ext.D_cba is not None:
        mat_pba_se = aggregate_material_categories(mat_ext.D_pba.loc[:, SWEDEN_CODE])
        mat_cba_se = aggregate_material_categories(mat_ext.D_cba.loc[:, SWEDEN_CODE])

        results["material"] = {
            "pba_se": mat_pba_se.sum(axis=1),
            "cba_se": mat_cba_se.sum(axis=1),
            "D_pba_se": mat_ext.D_pba.loc[:, SWEDEN_CODE],
            "D_cba_se": mat_ext.D_cba.loc[:, SWEDEN_CODE],
            "D_cba_full": mat_ext.D_cba,
            "D_pba_full": mat_ext.D_pba,
            "unit": "kt",
        }
        log.info("Extracted material results")
    else:
        log.warning("material extension D_cba not available")

    # ------------------------------------------------------------------
    # 2) Air emissions / GHG
    # ------------------------------------------------------------------
    air_ext = getattr(exio, "air_emissions", None)
    if air_ext is not None and air_ext.D_cba is not None:
        ghg_pba_se = aggregate_ghg(air_ext.D_pba.loc[:, SWEDEN_CODE], verbose=True)
        ghg_cba_se = aggregate_ghg(air_ext.D_cba.loc[:, SWEDEN_CODE], verbose=True)

        results["ghg"] = {
            "pba_se": ghg_pba_se.sum(axis=1),
            "cba_se": ghg_cba_se.sum(axis=1),
            "D_pba_se": air_ext.D_pba.loc[:, SWEDEN_CODE],
            "D_cba_se": air_ext.D_cba.loc[:, SWEDEN_CODE],
            "D_cba_full": air_ext.D_cba,
            "D_pba_full": air_ext.D_pba,
            "unit": "kt CO2e",
        }
        log.info("Extracted GHG results")
    else:
        log.warning("air_emissions extension D_cba not available")

    # ------------------------------------------------------------------
    # 3) Economic value / factor inputs
    # ------------------------------------------------------------------
    fi_ext = getattr(exio, "factor_inputs", None)
    if fi_ext is not None and fi_ext.D_cba is not None:
        results["factor_inputs"] = {
            "pba_se_total": fi_ext.D_pba.loc[:, SWEDEN_CODE].sum().sum(),
            "cba_se_total": fi_ext.D_cba.loc[:, SWEDEN_CODE].sum().sum(),
            "D_pba_se_by_sector": fi_ext.D_pba.loc[:, SWEDEN_CODE].sum(axis=0),
            "D_cba_full": fi_ext.D_cba,
            "D_pba_full": fi_ext.D_pba,
            "unit": "M.EUR",
        }
        log.info("Extracted factor_inputs results")
    else:
        log.warning("factor_inputs extension D_cba not available")

    return results


results = extract_sweden_results(exio)

In [ ]:

def top_sectors(results, n=5):
    """Print the top Sweden sectors by PBA and CBA for the three dimensions."""
    print("=" * 70)
    print("TOP SECTORS — Sweden")
    print("=" * 70)
    print()

    top_results = {}

    # ------------------------------------------------------------------
    # Material extraction
    # ------------------------------------------------------------------
    if "material" in results:
        print("─" * 70)
        print("DIMENSION 1 — Material extraction (kt)")
        print("─" * 70)

        D_pba = results["material"]["D_pba_se"]
        D_cba = results["material"]["D_cba_se"]

        pba_by_sector = aggregate_material_categories(D_pba).sum(axis=0)
        cba_by_sector = aggregate_material_categories(D_cba).sum(axis=0)

        pba_top = pba_by_sector.sort_values(ascending=False).head(n)
        cba_top = cba_by_sector.sort_values(ascending=False).head(n)

        print(f"\n  Top {n} sectors by PBA (Swedish production hotspots):")
        print(f"  {'Rank':<5} {'Sector':<65} {'kt':>10}")
        print(f"  {'-'*5} {'-'*65} {'-'*10}")
        for rank, (sector, val) in enumerate(pba_top.items(), 1):
            print(f"  {rank:<5} {str(sector):<65} {val:>10,.1f}")

        print(f"\n  Top {n} sectors by CBA (Swedish consumption hotspots):")
        print(f"  {'Rank':<5} {'Sector':<65} {'kt':>10}")
        print(f"  {'-'*5} {'-'*65} {'-'*10}")
        for rank, (sector, val) in enumerate(cba_top.items(), 1):
            print(f"  {rank:<5} {str(sector):<65} {val:>10,.1f}")

        top_results["material"] = {"pba_top": pba_top, "cba_top": cba_top, "unit": "kt"}

    # ------------------------------------------------------------------
    # GHG emissions
    # ------------------------------------------------------------------
    if "ghg" in results:
        print("\n" + "─" * 70)
        print("DIMENSION 2 — GHG emissions (kt CO2e)")
        print("─" * 70)

        D_pba = results["ghg"]["D_pba_se"]
        D_cba = results["ghg"]["D_cba_se"]

        pba_by_sector = aggregate_ghg(D_pba).sum(axis=0)
        cba_by_sector = aggregate_ghg(D_cba).sum(axis=0)

        pba_top = pba_by_sector.sort_values(ascending=False).head(n)
        cba_top = cba_by_sector.sort_values(ascending=False).head(n)

        print(f"\n  Top {n} sectors by PBA (Swedish production hotspots):")
        print(f"  {'Rank':<5} {'Sector':<65} {'kt CO2e':>10}")
        print(f"  {'-'*5} {'-'*65} {'-'*10}")
        for rank, (sector, val) in enumerate(pba_top.items(), 1):
            print(f"  {rank:<5} {str(sector):<65} {val:>10,.2f}")

        print(f"\n  Top {n} sectors by CBA (Swedish consumption hotspots):")
        print(f"  {'Rank':<5} {'Sector':<65} {'kt CO2e':>10}")
        print(f"  {'-'*5} {'-'*65} {'-'*10}")
        for rank, (sector, val) in enumerate(cba_top.items(), 1):
            print(f"  {rank:<5} {str(sector):<65} {val:>10,.2f}")

        top_results["ghg"] = {"pba_top": pba_top, "cba_top": cba_top, "unit": "kt CO2e"}

    # ------------------------------------------------------------------
    # Factor inputs
    # ------------------------------------------------------------------
    if "factor_inputs" in results:
        print("\n" + "─" * 70)
        print("DIMENSION 3 — Economic value (M.EUR)")
        print("─" * 70)

        D_pba = results["factor_inputs"]["D_pba_full"].loc[:, SWEDEN_CODE]
        fp_fi_tot = D_pba.sum(axis=0)
        top_pairs = fp_fi_tot.sort_values(ascending=False).head(n)
        top_results["factor_inputs"] = {"top_pairs": top_pairs, "unit": "M.EUR"}

        print(f"  {'Rank':<5} {'Country':<25} {'Sector':<40} {'M.EUR':>10}")
        for rank, (idx, val) in enumerate(top_pairs.items(), 1):
            print(f"  {rank:<5} {country_name(idx[0]):<25} {str(idx[1])[:40]:<40} {val:>10,.1f}")

    return top_results


top = top_sectors(results, n=5)


In [ ]:

def top_source_country_sectors(results, n=10):
    """Print the top source country-sector pairs for Sweden CBA footprints."""
    print("=" * 70)
    print("TOP SOURCE COUNTRY-SECTORS — Sweden CBA footprint")
    print("=" * 70)

    cs = {}

    # ------------------------------------------------------------------
    # Material
    # ------------------------------------------------------------------
    if "material" in results:
        print("\nDIMENSION 1 — Material (kt)")
        fp_mat = results["material"]["D_cba_full"].loc[:, SWEDEN_CODE]
        fp_mat_agg = aggregate_material_categories(fp_mat)
        fp_mat_tot = fp_mat_agg.sum(axis=0)
        top_pairs = fp_mat_tot.sort_values(ascending=False).head(n)
        cs["material"] = {"top_pairs": top_pairs, "unit": "kt"}

        print(f"  {'Rank':<5} {'Country':<30} {'Sector':<60} {'kt':>10}")
        for rank, (idx, val) in enumerate(top_pairs.items(), 1):
            print(f"  {rank:<5} {country_name(idx[0]):<30} {str(idx[1]):<60} {val:>10,.1f}")

    # ------------------------------------------------------------------
    # GHG
    # ------------------------------------------------------------------
    if "ghg" in results:
        print("\nDIMENSION 2 — GHG (kt CO2e)")
        fp_ghg = results["ghg"]["D_cba_full"].loc[:, SWEDEN_CODE]
        fp_ghg_agg = aggregate_ghg(fp_ghg)
        fp_ghg_tot = fp_ghg_agg.sum(axis=0)
        top_pairs = fp_ghg_tot.sort_values(ascending=False).head(n)
        cs["ghg"] = {"top_pairs": top_pairs, "unit": "kt CO2e"}

        print(f"  {'Rank':<5} {'Country':<30} {'Sector':<60} {'kt CO2e':>10}")
        for rank, (idx, val) in enumerate(top_pairs.items(), 1):
            print(f"  {rank:<5} {country_name(idx[0]):<30} {str(idx[1]):<60} {val:>10,.2f}")

    # ------------------------------------------------------------------
    # Factor inputs
    # ------------------------------------------------------------------
    if "factor_inputs" in results:
        print("\nDIMENSION 3 — Economic value (M.EUR)")
        fp_fi = results["factor_inputs"]["D_cba_full"].loc[:, SWEDEN_CODE]
        fp_fi_tot = fp_fi.sum(axis=0)
        top_pairs = fp_fi_tot.sort_values(ascending=False).head(n)
        cs["factor_inputs"] = {"top_pairs": top_pairs, "unit": "M.EUR"}

        print(f"  {'Rank':<5} {'Country':<30} {'Sector':<60} {'M.EUR':>10}")
        for rank, (idx, val) in enumerate(top_pairs.items(), 1):
            print(f"  {rank:<5} {country_name(idx[0]):<30} {str(idx[1]):<60} {val:>10,.1f}")

    return cs


cs = top_source_country_sectors(results, n=10)


In [ ]:
# Summary tables

summary = {}

if "material" in results:
    summary["material"] = pd.DataFrame({
        "PBA": results["material"]["pba_se"],
        "CBA": results["material"]["cba_se"],
    })
    summary["material"]["Net"] = summary["material"]["PBA"] - summary["material"]["CBA"]

if "ghg" in results:
    summary["ghg"] = pd.DataFrame({
        "PBA": results["ghg"]["pba_se"],
        "CBA": results["ghg"]["cba_se"],
    })
    summary["ghg"]["Net"] = summary["ghg"]["PBA"] - summary["ghg"]["CBA"]

if "factor_inputs" in results:
    summary["factor_inputs"] = pd.DataFrame({
        "value": [results["factor_inputs"]["pba_se_total"], results["factor_inputs"]["cba_se_total"]],
    }, index=["PBA", "CBA"])
    summary["factor_inputs"]["Net"] = np.nan

for dim, df in summary.items():
    print(f"\n{dim.upper()}")
    display(df)

In [ ]:
# Sanity checks and compact workshop-style printout

print("=" * 70)
print("WORKSHOP SUMMARY: Sweden results")
print("=" * 70)
print(f"Base year : {BASE_YEAR}")
print()

if "material" in results:
    print("─" * 70)
    print("DIMENSION 1 — Material extraction (kt)")
    print("─" * 70)
    print(f"  {'Category':>12}  {'PBA':>12}  {'CBA':>12}  {'Net':>12}  Direction")
    print(f"  {'-'*12}  {'-'*12}  {'-'*12}  {'-'*12}  {'-'*13}")
    for cat in results["material"]["pba_se"].index:
        pba = results["material"]["pba_se"][cat]
        cba = results["material"]["cba_se"][cat]
        net = pba - cba
        direction = "net exporter" if net > 0 else "NET IMPORTER"
        print(f"  {cat:>12}  {pba:>12,.1f}  {cba:>12,.1f}  {net:>12,.1f}  {direction}")
    print()

if "ghg" in results:
    print("─" * 70)
    print("DIMENSION 2 — GHG emissions (kt CO2e)")
    print("─" * 70)
    print(f"  {'Category':>22}  {'PBA':>12}  {'CBA':>12}  {'Net':>12}  Direction")
    print(f"  {'-'*22}  {'-'*12}  {'-'*12}  {'-'*12}  {'-'*13}")
    for cat in results["ghg"]["pba_se"].index:
        pba = results["ghg"]["pba_se"][cat]
        cba = results["ghg"]["cba_se"][cat]
        net = pba - cba
        direction = "net exporter" if net > 0 else "NET IMPORTER"
        print(f"  {cat:>22}  {pba:>12,.1f}  {cba:>12,.1f}  {net:>12,.1f}  {direction}")
    print()

if "factor_inputs" in results:
    print("─" * 70)
    print("DIMENSION 3 — Economic value: total factor inputs (M.EUR)")
    print("─" * 70)
    pba_total = results["factor_inputs"]["pba_se_total"]
    cba_total = results["factor_inputs"]["cba_se_total"]
    print(f"  PBA (Swedish production value) : {pba_total:>12,.1f} M.EUR")
    print(f"  CBA (Swedish consumption value): {cba_total:>12,.1f} M.EUR")
    print(f"  Net (PBA - CBA)                : {pba_total - cba_total:>12,.1f} M.EUR")
    print()

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

CAT_COLORS = {
    "biomass":  "#55A868",
    "fossil":   "#C44E52",
    "metals":   "#8172B2",
    "minerals": "#937860",
}

def add_note(ax, note="Hotspot analysis for Sweden"):
    ax.text(
        0.99, 0.01, note, transform=ax.transAxes,
        fontsize=7, color='grey', ha='right', va='bottom', style='italic'
    )

def country_sector_chart(top_pairs, xlabel, title, filename):
    labels = [format_country_sector_pair(idx) for idx in top_pairs.index]
    vals = top_pairs.values
    unique_countries = list(dict.fromkeys(idx[0] for idx in top_pairs.index))
    cmap = plt.colormaps.get_cmap('tab10').resampled(max(len(unique_countries), 1))
    c_map = {c: cmap(i) for i, c in enumerate(unique_countries)}
    colors = [c_map[idx[0]] for idx in top_pairs.index]

    fig, ax = plt.subplots(figsize=(12, max(6, 0.42 * len(labels) + 2)))
    y = np.arange(len(labels))
    ax.barh(y, vals, color=colors, edgecolor='white', height=0.65)
    ax.set_yticks(y)
    ax.set_yticklabels([fill(label, width=55) for label in labels], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=11)

    patches = [mpatches.Patch(color=c_map[c], label=country_name(c))
               for c in unique_countries]
    ax.legend(handles=patches, loc='lower right', fontsize=8)

    for i, v in enumerate(vals):
        ax.text(v * 1.01 if v >= 0 else v * 0.99, i, f"{v:,.1f}", va='center', fontsize=8)

    ax.grid(axis='x', alpha=0.25)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches='tight')
    plt.close(fig)

if "material" in top:
    country_sector_chart(
        top["material"]["cba_top"],
        "kt",
        "Top Swedish material consumption hotspots by source country-sector",
        "country_sector_material_cba.png",
    )

if "ghg" in top:
    country_sector_chart(
        top["ghg"]["cba_top"],
        "kt CO2e",
        "Top Swedish GHG consumption hotspots by source country-sector",
        "country_sector_ghg_cba.png",
    )

if "factor_inputs" in top:
    country_sector_chart(
        top["factor_inputs"]["top_pairs"],
        "M.EUR",
        "Top Swedish economic value source country-sectors",
        "country_sector_factor_inputs_cba.png",
    )


In [ ]:
# Save compact CSV outputs

log.info("Saving CSV outputs ...")

for dim, data in top.items():
    unit = results[dim]["unit"]
    data["pba_top"].to_frame(f"PBA_Sweden_{unit}").to_csv(OUTPUT_DIR / f"top_sectors_PBA_{dim}.csv")
    data["cba_top"].to_frame(f"CBA_Sweden_{unit}").to_csv(OUTPUT_DIR / f"top_sectors_CBA_{dim}.csv")

for dim, data in cs.items():
    unit = data["unit"]
    data["top_pairs"].to_frame(f"CBA_top_sectors_{unit}").to_csv(OUTPUT_DIR / f"top_country_sectors_CBA_{dim}.csv")

for dim, df in summary.items():
    df.to_csv(OUTPUT_DIR / f"summary_{dim}.csv")

log.info(f"All outputs written to: {OUTPUT_DIR}")
print(f"Outputs written to: {OUTPUT_DIR.resolve()}")